In [9]:
import itertools
import random
import csv

In [10]:
def write_csv_dict(filename, data):
    with open(filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data[0].keys())
        writer.writeheader()
        writer.writerows(data)

In [ ]:
class AgreementTemplateGenerator:
    def __init__(self):
        self.subjects = {
            'sg': '[wug]',
            'pl': '[wugs]'
        }
        
        # All verbs now uniformly use the 'comp' key
        self.verbs = [
            # Auxiliaries / Copulas
            {'sg': 'is', 'pl': 'are', 'comp': 'happy'},
            {'sg': 'was', 'pl': 'were', 'comp': 'sleeping'},
            {'sg': 'has', 'pl': 'have', 'comp': 'arrived'},
            {'sg': 'does', 'pl': 'do', 'comp': 'understand'},
            
            # Lexical Verbs (can add more)
            {'sg': 'jumps', 'pl': 'jump', 'comp': 'high'},      # Added adverb
            {'sg': 'opens', 'pl': 'open', 'comp': 'doors'}, 
            {'sg': 'runs', 'pl': 'run', 'comp': 'fast'},    
            {'sg': 'eats', 'pl': 'eat', 'comp': 'apples'},      # Added object
            {'sg': 'sleeps', 'pl': 'sleep', 'comp': 'soundly'}, # Added adverb
            {'sg': 'sings', 'pl': 'sing', 'comp': 'loudly'}
        ]
        
        # can add more here
        self.distractors = [
            {'sg': 'tree', 'pl': 'trees'},
            {'sg': 'car', 'pl': 'cars'},
            {'sg': 'box', 'pl': 'boxes'},
            {'sg': 'chair', 'pl': 'chairs'},
            {'sg': 'bridge', 'pl': 'bridges'}
        ]
        
        self.prepositions = [
            "near the",
            "behind the",
            "under the",
            "beside the",
            "in front of the",
            "next to the",
            "past the"
        ]

    def format_sentence(self, subj, prep_phrase, verb, comp):
        """Builds the sentence uniformly for all verb types."""
        prep_str = f" {prep_phrase}" if prep_phrase else ""
        comp_str = f" {comp}" if comp else ""
        
        raw_sentence = f"The {subj}{prep_str} {verb}{comp_str}"
        return f"{raw_sentence.strip()}."

    def generate_templates(self):
        items = []
        
        for verb_dict in self.verbs:
            comp = verb_dict['comp']
            
            for i, distractor in enumerate(self.distractors):
                prep1 = self.prepositions[i % len(self.prepositions)]
                prep2 = self.prepositions[(i + 1) % len(self.prepositions)]
                
                # -------------------------
                # 0 Distractors
                # -------------------------
                for subj_num in ['sg', 'pl']:
                    subj = self.subjects[subj_num]
                    good_verb = verb_dict[subj_num]
                    bad_verb = verb_dict['pl' if subj_num == 'sg' else 'sg']
                    
                    items.append({
                        'distractors': 0,
                        'condition': f"subj_{subj_num}",
                        'good': self.format_sentence(subj, "", good_verb, comp),
                        'bad': self.format_sentence(subj, "", bad_verb, comp)
                    })
                
                # -------------------------
                # 1 Distractor
                # -------------------------
                for subj_num, d1_num in itertools.product(['sg', 'pl'], ['sg', 'pl']):
                    subj = self.subjects[subj_num]
                    d1 = distractor[d1_num]
                    good_verb = verb_dict[subj_num]
                    bad_verb = verb_dict['pl' if subj_num == 'sg' else 'sg']
                    
                    prep_phrase = f"{prep1} {d1}"
                    
                    items.append({
                        'distractors': 1,
                        'condition': f"subj_{subj_num}_d1_{d1_num}",
                        'good': self.format_sentence(subj, prep_phrase, good_verb, comp),
                        'bad': self.format_sentence(subj, prep_phrase, bad_verb, comp)
                    })
                
                # -------------------------
                # 2 Distractors
                # -------------------------
                second_distractor = self.distractors[(i + 1) % len(self.distractors)]
                
                for subj_num, d1_num, d2_num in itertools.product(['sg', 'pl'], ['sg', 'pl'], ['sg', 'pl']):
                    subj = self.subjects[subj_num]
                    d1 = distractor[d1_num]
                    d2 = second_distractor[d2_num]
                    good_verb = verb_dict[subj_num]
                    bad_verb = verb_dict['pl' if subj_num == 'sg' else 'sg']
                    
                    prep_phrase = f"{prep1} {d1} and {prep2} {d2}"
                    
                    items.append({
                        'distractors': 2,
                        'condition': f"subj_{subj_num}_d1_{d1_num}_d2_{d2_num}",
                        'good': self.format_sentence(subj, prep_phrase, good_verb, comp),
                        'bad': self.format_sentence(subj, prep_phrase, bad_verb, comp)
                    })
                    
        return items

In [ ]:
# Generate dataset
generator = AgreementTemplateGenerator()
dataset = generator.generate_templates()

In [7]:
# add idx for each item
for idx, item in enumerate(dataset):
    item['idx'] = idx

In [ ]:
# format so that idx is first column
dataset = [{k: item[k] for k in ['idx', 'distractors', 'condition', 'good', 'bad']} for item in dataset]

In [17]:
write_csv_dict('agreement_stimuli.csv', dataset)